# 06. Evaluation and tuning

A forecast is only useful if you know how wrong it tends to be. This notebook measures accuracy on data the model has not seen, compares a single backtest with rolling cross-validation, and shows what automatic tuning can and cannot do.

- Backtest: one held-out period
- Cross-validation: many held-out periods
- Choosing a metric
- Tuning with `tune=True`
- Did tuning help? A fair comparison

Previous: [05. Uncertainty and outliers](05_uncertainty_and_outliers.ipynb) | Next: [07. Reports and dashboards](07_reports_and_dashboards.ipynb)

In [ ]:
import dataclasses

import pandas as pd

from prophet import forecast, tune
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.visuals import backtest_metrics

from helpers import load_example, quiet_logs

quiet_logs()

## Backtest: one held-out period

`result.backtest()` holds back the end of the history, refits the same model on everything before it and compares the forecast with what actually happened. It is the quickest honest check, and the default holdout is the forecast's own horizon.

In [ ]:
df = load_example("peyton_manning")
# The series has a few missing days; freq="D" says it is daily anyway.
result = forecast(df, horizon="365D", freq="D")
bt = result.backtest()
pd.Series(dataclasses.asdict(bt.metrics)).round(3)

In [ ]:
fig = result.plot_backtest(backtest=bt)

The title states the typical error and how many actuals landed inside the 80% band. A single period can be lucky or unlucky, though, which is what cross-validation addresses.

## Cross-validation: many held-out periods

Cross-validation repeats the backtest from several cutoffs: train on an initial window, forecast a horizon, move the cutoff forward by a period and repeat. It shows how error grows with the forecast horizon.

In [ ]:
df_cv = cross_validation(
    result.model,
    initial="730 days",
    period="180 days",
    horizon="365 days",
    disable_tqdm=True,
)
df_cv["cutoff"].nunique(), len(df_cv)

In [ ]:
metrics = performance_metrics(df_cv)
metrics[["horizon", "mae", "rmse", "mape", "coverage"]].iloc[::60]

In [ ]:
fig = plot_cross_validation_metric(df_cv, metric="mape")

Error rises gently with the horizon. Use a single backtest for a quick sanity check, and cross-validation when you need to know how accuracy changes with how far ahead you look, or when one period could be unrepresentative. On long series or with many folds, `parallel="processes"` fits the folds on all CPU cores.

## Choosing a metric

- **MAE**: the average miss in the units of the data. Easy to explain.
- **RMSE**: like MAE but punishes large misses more. Use it when big errors are costly.
- **MAPE**: the average miss as a share of the actual value. Comparable across series, but meaningless when values are near zero (the reports switch to MAE automatically then).
- **Coverage**: the share of actuals inside the uncertainty band. For an 80% band it should be near 0.8; Prophet's intervals often come in lower, as notebook 05 showed.

In [ ]:
metrics[["mae", "rmse", "mape", "coverage"]].mean().round(3)

## Tuning with `tune=True`

The two prior scales and the seasonality mode matter most. `tune=True` scores a grid of 33 combinations, including Prophet's defaults, by cross-validation in parallel and fits the best one. Air Passengers is a good case: its seasonal swings grow with the trend, which the default additive mode misses.

In [ ]:
air = load_example("air_passengers")
tuned = forecast(air, horizon="24 months", tune=True)
tuned.tuning.best_params

In [ ]:
cols = ["changepoint_prior_scale", "seasonality_prior_scale", "seasonality_mode", "rmse", "rank"]
tuned.tuning.scores[cols].head()

`prophet.tune(df)` runs the same search on its own and returns the result without fitting a forecast, which is handy when you want to inspect the scores first. `best_model()` then fits the winner. In a script, call tuning under `if __name__ == "__main__":`, because it starts worker processes.

In [ ]:
search = tune(air, param_grid={"seasonality_mode": ["additive", "multiplicative"]})
search.scores[["seasonality_mode", "rmse", "rank"]]

### Searching ranges with Optuna

A grid only tries the values you list, and it grows with every parameter you add. For settings that are naturally continuous, such as the prior scales, `search="optuna"` is usually the better fit: you give a range, and Optuna's TPE sampler uses the scores so far to choose which values to try next. Prefer it when a parameter is a range rather than a handful of values, or when the space is too large for a grid. Stay with the grid for a few short lists, where it tries every combination.

Each entry in `param_grid` is a `FloatRange`, an `IntRange` or a list of categorical choices, and `n_trials` sets the budget. `log=True` samples on a log scale, which suits prior scales that span orders of magnitude. The cross-validation windows and the metric are the same as for the grid, so the scores can be compared. Optuna is an optional extra, so install it first with `pip install "neoprophet[optuna]"`.

In [ ]:
from prophet import FloatRange

optuna_search = tune(
    air,
    search="optuna",
    n_trials=30,
    param_grid={
        "changepoint_prior_scale": FloatRange(0.001, 0.5, log=True),
        "seasonality_prior_scale": FloatRange(0.01, 10.0, log=True),
        "seasonality_mode": ["additive", "multiplicative"],
    },
)

In [ ]:
best = optuna_search.best_params
print({name: round(v, 4) if isinstance(v, float) else v for name, v in best.items()})

cols = list(best) + ["rmse", "rank", "trial", "pruned"]
optuna_search.scores[cols].head()

`trial` is the Optuna trial number and `pruned` flags trials that were stopped early. Nothing is pruned here: pass `prune=True` (with `parallel=False`) to stop weak trials part-way through cross-validation. Prophet's defaults are scored as one extra row (`is_default`), so you can see whether the search beat them. `optuna_search.study` is the `optuna.Study` behind the search, so Optuna's own tools work on it, for example `optuna.visualization.plot_param_importances(optuna_search.study)` (needs plotly). `forecast(air, horizon="24 months", tune="optuna")` runs the same kind of search over the default ranges and fits the winner.

## Did tuning help? A fair comparison

To judge tuning fairly, hold the last period out before tuning, so the search never sees it, then compare tuned and default forecasts on that period. The function below does exactly that.

In [ ]:
def tuned_vs_default(df: pd.DataFrame, holdout: int, freq: str | None = None) -> pd.DataFrame:
    """Holdout error of default and tuned forecasts, with tuning kept away from the holdout."""
    train, test = df.iloc[:-holdout], df.iloc[-holdout:]
    # A calendar horizon covers the whole holdout even when the data has gaps.
    span = f"{(test['ds'].max() - train['ds'].max()).days}D"
    rows = {}
    for name, tune_flag in [("default", False), ("tuned", True)]:
        res = forecast(train, horizon=span, freq=freq, tune=tune_flag)
        both = test.merge(res.forecast, on="ds")
        m = backtest_metrics(
            both["y"].to_numpy(), both["yhat"].to_numpy(),
            both["yhat_lower"].to_numpy(), both["yhat_upper"].to_numpy(),
            interval_width=res.model.interval_width,
        )
        rows[name] = {"mae": m.mae, "rmse": m.rmse, "mape": m.mape, "coverage": m.coverage}
    return pd.DataFrame(rows).T.round(3)


air_table = tuned_vs_default(air, holdout=24)
air_table

In [ ]:
peyton_table = tuned_vs_default(load_example("peyton_manning"), holdout=365, freq="D")
peyton_table

In [ ]:
for name, table in [("Air Passengers", air_table), ("Peyton Manning", peyton_table)]:
    change = table.loc["tuned", "rmse"] / table.loc["default", "rmse"] - 1
    verdict = "better" if change < -0.02 else "worse" if change > 0.02 else "about the same"
    direction = "lower" if change < 0 else "higher"
    print(f"{name}: tuned RMSE is {abs(change):.0%} {direction} than the default, so {verdict}.")

Cross-validation ranked the tuned settings best on past folds, but on a truly unseen period they need not beat the defaults, as the verdicts above show. Cross-validation rewards settings that suited the past, so it misleads when the latest period behaves differently, and a short holdout such as 24 months is noisy. Treat tuning as a starting point and check it on a held-out period like this before trusting it.

## Next

[07. Reports and dashboards](07_reports_and_dashboards.ipynb) turns a forecast into something you can share: a one-page report, an offline dashboard and saved models.